# Flow Matching on CIFAR-10 — main training notebook

Reproduction réduite de **Lipman et al., *Flow Matching for Generative Modeling*** (ICLR 2023). Ce notebook entraîne **un** modèle. Pour reproduire la Table 1 du papier, l'exécuter trois fois en changeant `PATH_TYPE` dans la cellule de config :

| `PATH_TYPE` | Modèle | Loss | Cible de régression |
|---|---|---|---|
| `"ot"`   | FM avec chemin Optimal Transport | CFM | $u_t(x \mid x_1) = x_1 - (1-\sigma_{\min})x_0$ |
| `"vp"`   | FM avec chemin VP-Diffusion       | CFM | $u_t(x \mid x_1)$ analytique (Th. 3) |
| `"ddpm"` | DDPM (Ho 2020) baseline           | $\epsilon$-matching | $x_0$ (le bruit) |

**Hardware** : conçu pour Colab A100/H100. ~30h par run à 391k steps. Reprise automatique depuis le dernier checkpoint sur Google Drive.

## 1. Configuration

**Change uniquement `PATH_TYPE` entre les 3 runs.** Le reste suit la lignée ADM/Dhariwal-Nichol (le papier sous-spécifie CIFAR-10). Le batch effectif reste 256 via accumulation de gradient.

In [ ]:
PATH_TYPES = ["ot", "vp", "ddpm"]
MAX_STEPS = 391_000 
BATCH_SIZE = 64     
ACCUM_ITER = 4          
USE_WANDB = False
DRIVE_ROOT = "/content/drive/MyDrive/Conditional Flow Matching/flow-matching-b3"  
REPO_URL = "https://github.com/Alexis5131/Conditional-Flow-Matching.git"

## 2. Colab setup (skipped on local)

In [ ]:
import os, sys, subprocess, shutil
IN_COLAB = "google.colab" in sys.modules
print("Colab:", IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    drive_repo_path = "/content/drive/MyDrive/Conditional Flow Matching /flow-matching-b3"
    local_repo_path = "/content/flow-matching-b3"

    if not os.path.exists(local_repo_path):
        print("Copie du dépôt depuis Google Drive...")
        shutil.copytree(drive_repo_path, local_repo_path)

    subprocess.run(["pip", "install", "-q", "-e", local_repo_path], check=True)
    sys.path.insert(0, f"{local_repo_path}/src")
else:
    sys.path.insert(0, os.path.abspath("../src"))

In [ ]:
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("vram :", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

## 3. Eval helper — FID (`legacy_tensorflow`, protocole papier)

Le mid-training utilise 10k échantillons (biaisé ↑ vs les 50k de la table finale), tous les `eval_every` steps.

In [ ]:
from pathlib import Path
import torch
from flow_matching_b3.paths import get_path
from flow_matching_b3.sampling import sample
from flow_matching_b3.fid import compute_fid_cifar10
from flow_matching_b3.train import denorm

def make_fid_fn(path, n_samples=10_000, batch=128, solver="dopri5"):
    @torch.no_grad()
    def _fid(model):
        def gen():
            remaining = n_samples
            while remaining > 0:
                b = min(batch, remaining)
                imgs, _ = sample(model, path, shape=(b, 3, 32, 32), solver=solver, device=next(model.parameters()).device)
                yield denorm(imgs)
                remaining -= b
        return compute_fid_cifar10(gen(), n_samples=n_samples)
    return _fid

## 4. Training

In [ ]:
from pathlib import Path
import os
import torch
import matplotlib.pyplot as plt

from flow_matching_b3.train import TrainConfig, train, find_latest_ckpt, make_grid_png
from flow_matching_b3.paths import get_path
from flow_matching_b3.unet import adm_unet_cifar10
from flow_matching_b3.ema import EMA
from flow_matching_b3.sampling import sample

if USE_WANDB:
    import wandb
    wandb.login()

for pt in PATH_TYPES:
    print(f"\n{'='*50}\nLANCEMENT DE L'ENTRAÎNEMENT POUR PATH_TYPE = {pt}\n{'='*50}")

    current_run_name = f"fm-{pt}-cifar10"
    current_run_dir = f"{DRIVE_ROOT}/{current_run_name}" if IN_COLAB else f"./runs/{current_run_name}"
    os.makedirs(current_run_dir, exist_ok=True)

    cfg = TrainConfig(
        path_type=pt,
        run_dir=Path(current_run_dir),
        data_root=Path("/content/data") if IN_COLAB else Path("./data"),
        max_steps=MAX_STEPS,
        batch_size=BATCH_SIZE,
        accum_iter=ACCUM_ITER,
        lr_peak=1e-4,
        warmup_steps=45_000,
        poly_decay=True,
        ema_decay=0.9999,
        ckpt_every=25_000,
        log_every=200,
        eval_every=50_000,
        eval_n_samples=10_000,
        device="cuda" if torch.cuda.is_available() else "cpu",
        dtype="float32",
        use_wandb=USE_WANDB,
        wandb_run_name=current_run_name,
    )

    if cfg.path_type == "ot":
        eval_path = get_path("ot", sigma_min=cfg.sigma_min)
    elif cfg.path_type == "vp":
        eval_path = get_path("vp", beta_min=cfg.vp_beta_min, beta_max=cfg.vp_beta_max)
    else:
        eval_path = get_path("ddpm", beta_min=cfg.vp_beta_min, beta_max=cfg.vp_beta_max)

    fid_fn = make_fid_fn(eval_path, n_samples=cfg.eval_n_samples)

    final_ckpt = train(cfg, fid_fn=fid_fn)
    print(f"\nEntraînement terminé pour {pt}. Checkpoint: {final_ckpt}")

    model = adm_unet_cifar10().to(cfg.device)
    ema = EMA(model, decay=cfg.ema_decay)
    state = torch.load(find_latest_ckpt(Path(current_run_dir)), map_location=cfg.device)
    model.load_state_dict(state["model"])
    ema.load_state_dict(state["ema"])

    with ema.swap_into(model):
        model.eval()
        final_fid = make_fid_fn(eval_path, n_samples=50_000)(model)
        print(f"FINAL FID(EMA, 50k) for {pt}: {final_fid:.3f}")
        grid_samples, _ = sample(model, eval_path, shape=(64, 3, 32, 32), solver="dopri5", device=cfg.device, seed=0)

    grid = make_grid_png(grid_samples, n_row=8).permute(1, 2, 0).cpu()
    plt.figure(figsize=(8, 8))
    plt.imshow(grid)
    plt.axis("off")
    plt.title(f"{current_run_name} — FID(50k) = {final_fid:.2f}")
    plt.savefig(f"{current_run_dir}/final_samples.png", bbox_inches="tight", dpi=120)
    plt.show()

    del model, ema, state, cfg, final_ckpt
    torch.cuda.empty_cache()